In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib as plt
import matplotlib.pyplot as plot
from 

# ── Load MNIST properly this time — torchvision ───────────
# torchvision handles download, normalisation, batching
transform = transforms.Compose([
    transforms.ToTensor(),                    # PIL → tensor, 0-1
    transforms.Normalize((0.1307,), (0.3081,)) # MNIST mean and std
])

train_dataset = datasets.MNIST('./data', train=True,
                                download=True, transform=transform)
test_dataset  = datasets.MNIST('./data', train=False,
                                transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=256)


# ── Training functions ────────────────────────────────────
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for X_batch, y_batch in loader:
        optimizer.zero_grad()
        output = model(X_batch)
        loss   = criterion(output, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct    += (output.argmax(1) == y_batch).sum().item()
        total      += len(y_batch)

    return total_loss / len(loader), correct / total


def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            output   = model(X_batch)
            correct += (output.argmax(1) == y_batch).sum().item()
            total   += len(y_batch)
    return correct / total


# ── Training CNN ─────────────────────────────────────────────
cnn       = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cnn.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=2, factor=0.5
)

print("Training CNN on MNIST...\n")
best_acc = 0

for epoch in range(15):
    loss, train_acc = train_epoch(cnn, train_loader,
                                   criterion, optimizer)
    test_acc        = evaluate(cnn, test_loader)
    scheduler.step(test_acc)
    lr = optimizer.param_groups[0]['lr']

    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(cnn.state_dict(), 'best_cnn_mnist.pth')
        flag = " ← best"
    else:
        flag = ""

    print(f"Epoch {epoch+1:2d} | Loss: {loss:.4f} "
          f"| Train: {train_acc:.4f} | Test: {test_acc:.4f} "
          f"| LR: {lr:.6f}{flag}")

print(f"\nBest test accuracy: {best_acc:.4f}")


# ── Visualising what filters learned ───────────────────────
filters = cnn.conv1.weight.data   # shape (32, 1, 3, 3)

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(filters[i, 0].numpy(), cmap='gray')
    ax.axis('off')
plt.suptitle('32 learned filters — Conv layer 1', fontsize=12)
plt.tight_layout()
plt.show()